In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from torch.nn import CrossEntropyLoss
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
PROJECT_ROOT = Path("../..")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
NATURAL_CDS_FASTA = OUTPUT_DIR / "cds_real_histone.fasta"
NATURAL_PROTEIN_FASTA = OUTPUT_DIR / "protein_real_histone.fasta"

MODEL_NAME = "hugohrban/progen2-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 2048
BATCH_SIZE = 32
TARGET = "H1"

sns.set_theme(style="white")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"],
    "axes.linewidth": 1.2,
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "axes.unicode_minus": False,
})

DATASET_PALETTE = {
    "Natural Sequences": "#66c2a5",
    "Temperature 0.55 Top-p 0.7": "#fc8d62",
    "Temperature 0.7 Top-p 0.8": "#8da0cb",
    "Temperature 0.85 Top-p 0.9": "#e78ac3",
    "Temperature 1.0 Top-p 0.95": "#a6d854",
}

PLOT_SOURCE_ORDER = [
    "Natural Sequences",
    "Temperature 0.55 Top-p 0.7",
    "Temperature 0.7 Top-p 0.8",
    "Temperature 0.85 Top-p 0.9",
    "Temperature 1.0 Top-p 0.95",
]

HISTONE_TYPES = ["H1", "H2A", "H2B", "H3", "H4"]

PROTEIN_PATTERN = re.compile(r"^protein_temp_(?P<temp>[^_]+)_top_p_(?P<top_p>[^.]+)\\.fasta$")
CDS_PATTERN = re.compile(r"^cds_temp_(?P<temp>[^_]+)_top_p_(?P<top_p>[^.]+)\\.fasta$")

CODON_TABLE = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
    'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
    'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
    'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
    'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
    'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
    'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
    'TAC':'Y', 'TAT':'Y', 'TAA':'_', 'TAG':'_',
    'TGC':'C', 'TGT':'C', 'TGA':'_', 'TGG':'W',
}

AA_ORDER = ['F', 'L', 'I', 'M', 'V', 'S', 'P', 'T', 'A', 'Y', '_', 'H', 'Q', 'N', 'K', 'D', 'E', 'C', 'W', 'R', 'G']

In [ ]:
def set_nature_style() -> None:
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica"],
        "font.size": 14,
        "axes.titlesize": 18,
        "axes.labelsize": 14,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "axes.linewidth": 1.0,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 5.0,
        "ytick.major.size": 5.0,
    })


def calculate_gc3(seq_str: str) -> float:
    third_positions = seq_str[2::3]
    if not third_positions:
        return 0.0
    return (third_positions.count("G") + third_positions.count("C")) / len(third_positions) * 100


def identify_histone_type(description_text: str) -> str:
    desc_lower = description_text.lower()
    patterns = {
        "H1": r"histone[_ ]h1|linker histone",
        "H2A": r"histone[_ ]h2a",
        "H2B": r"histone[_ ]h2b",
        "H3": r"histone[_ ]h3",
        "H4": r"histone[_ ]h4",
    }
    for h_type, pattern in patterns.items():
        if re.search(pattern, desc_lower):
            return h_type
    return "Unknown"


def extract_temp_top_p(path: Path, pattern: re.Pattern[str]) -> tuple[str, str] | None:
    match = pattern.match(path.name)
    if match is None:
        return None
    return match.group("temp"), match.group("top_p")


def format_source_label(temp_str: str, top_p_str: str) -> str:
    return f"Temperature {temp_str.replace('p', '.')} Top-p {top_p_str.replace('p', '.')}"


def ordered_codons_and_boundaries() -> tuple[list[str], list[int]]:
    aa_to_codons = defaultdict(list)
    for codon, aa in CODON_TABLE.items():
        aa_to_codons[aa].append(codon)

    ordered_codons = []
    boundary_indices = []
    current_idx = 0
    for aa in AA_ORDER:
        codons = sorted(aa_to_codons[aa])
        ordered_codons.extend(codons)
        current_idx += len(codons)
        boundary_indices.append(current_idx)
    return ordered_codons, boundary_indices


ORDERED_CODONS, BOUNDARY_INDICES = ordered_codons_and_boundaries()

In [ ]:
def discover_generated_pairs(output_dir: Path) -> dict[str, dict[str, Path]]:
    pairs: dict[str, dict[str, Path]] = {}

    for protein_path in sorted(output_dir.glob("protein_temp_*_top_p_*.fasta")):
        values = extract_temp_top_p(protein_path, PROTEIN_PATTERN)
        if values is None:
            continue
        temp_str, top_p_str = values
        source = format_source_label(temp_str, top_p_str)
        pairs.setdefault(source, {})["protein"] = protein_path

    for cds_path in sorted(output_dir.glob("cds_temp_*_top_p_*.fasta")):
        values = extract_temp_top_p(cds_path, CDS_PATTERN)
        if values is None:
            continue
        temp_str, top_p_str = values
        source = format_source_label(temp_str, top_p_str)
        pairs.setdefault(source, {})["cds"] = cds_path

    return pairs


def build_manifest(output_dir: Path, natural_protein_fasta: Path, natural_cds_fasta: Path) -> pd.DataFrame:
    rows = [{
        "Source": "Natural Sequences",
        "Protein_Path": natural_protein_fasta,
        "CDS_Path": natural_cds_fasta,
    }]

    generated_pairs = discover_generated_pairs(output_dir)

    for source in PLOT_SOURCE_ORDER:
        if source == "Natural Sequences":
            continue
        if source in generated_pairs:
            rows.append({
                "Source": source,
                "Protein_Path": generated_pairs[source].get("protein"),
                "CDS_Path": generated_pairs[source].get("cds"),
            })

    for source in sorted(s for s in generated_pairs if s not in PLOT_SOURCE_ORDER):
        rows.append({
            "Source": source,
            "Protein_Path": generated_pairs[source].get("protein"),
            "CDS_Path": generated_pairs[source].get("cds"),
        })

    return pd.DataFrame(rows)


manifest = build_manifest(OUTPUT_DIR, NATURAL_PROTEIN_FASTA, NATURAL_CDS_FASTA)
manifest

In [ ]:
def load_cds_metrics(manifest: pd.DataFrame) -> pd.DataFrame:
    records = []

    for row in manifest.itertuples(index=False):
        cds_path = Path(row.CDS_Path) if row.CDS_Path else None
        if cds_path is None or not cds_path.exists():
            print(f"[Skip] CDS FASTA not found for {row.Source}: {cds_path}")
            continue

        count = 0
        for rec in SeqIO.parse(cds_path, "fasta"):
            seq_str = str(rec.seq).upper()
            if not seq_str:
                continue
            if not set(seq_str) <= set("ACGTN"):
                continue

            records.append({
                "Dataset": row.Source,
                "Histone_Type": identify_histone_type(rec.description),
                "ID": rec.id,
                "Sequence": seq_str,
                "Length": len(seq_str),
                "GC_Content": gc_fraction(seq_str) * 100,
                "GC3": calculate_gc3(seq_str),
            })
            count += 1

        print(f"[{row.Source}] CDS metrics loaded: {count}")

    return pd.DataFrame(records)


def load_protein_metrics(manifest: pd.DataFrame) -> pd.DataFrame:
    results = []

    for row in manifest.itertuples(index=False):
        protein_path = Path(row.Protein_Path) if row.Protein_Path else None
        if protein_path is None or not protein_path.exists():
            print(f"[Skip] Protein FASTA not found for {row.Source}: {protein_path}")
            continue

        count = 0
        for record in SeqIO.parse(protein_path, "fasta"):
            seq_str = str(record.seq).upper()
            if len(seq_str) < 30:
                continue
            if re.search(r"[XBZJOU]", seq_str):
                continue

            try:
                analysis = ProteinAnalysis(seq_str)
                results.append({
                    "Source": row.Source,
                    "ID": record.id,
                    "Sequence": seq_str,
                    "Length": len(seq_str),
                    "MW_kDa": analysis.molecular_weight() / 1000.0,
                    "pI": analysis.isoelectric_point(),
                    "Predicted_Type": identify_histone_type(record.description),
                    "Type": identify_histone_type(record.description),
                })
                count += 1
            except Exception:
                continue

        print(f"[{row.Source}] Protein metrics loaded: {count}")

    return pd.DataFrame(results)


df_cds = load_cds_metrics(manifest)
df_protein = load_protein_metrics(manifest)
display(df_cds.head())
display(df_protein.head())

In [ ]:
def load_progen2_model(model_name: str, device: str):
    print(f"Loading model: {model_name} on {device} ...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=torch.float16 if "cuda" in device else torch.float32,
        ).to(device)
        model.eval()
        return tokenizer, model
    except Exception as exc:
        print(f"Model loading failed: {exc}")
        return None, None


def calculate_batch_ppl(sequences, tokenizer, model, device, max_length=1024):
    inputs = tokenizer(sequences, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = input_ids[..., 1:].contiguous()
    shift_mask = attention_mask[..., 1:].contiguous()

    loss_fct = CrossEntropyLoss(reduction="none")
    flat_loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    per_token_loss = flat_loss.view(shift_labels.size())
    valid_loss = per_token_loss * shift_mask
    seq_len = shift_mask.sum(dim=1)
    per_seq_loss = valid_loss.sum(dim=1) / (seq_len + 1e-9)
    return torch.exp(per_seq_loss).cpu().numpy().tolist()


def build_ppl_dataframe(manifest: pd.DataFrame) -> pd.DataFrame:
    tokenizer, model = load_progen2_model(MODEL_NAME, DEVICE)
    if tokenizer is None or model is None:
        return pd.DataFrame()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    all_results = []

    for row in manifest.itertuples(index=False):
        protein_path = Path(row.Protein_Path) if row.Protein_Path else None
        if protein_path is None or not protein_path.exists():
            print(f"[Skip] Protein FASTA not found for {row.Source}: {protein_path}")
            continue

        raw_records = list(SeqIO.parse(protein_path, "fasta"))
        valid_records = []
        for rec in raw_records:
            seq_str = str(rec.seq).upper()
            if len(seq_str) < 30:
                continue
            if re.search(r"[XBZJOU]", seq_str):
                continue
            valid_records.append(rec)

        print(f"[{row.Source}] Valid sequences: {len(valid_records)} / {len(raw_records)}")

        for i in tqdm(range(0, len(valid_records), BATCH_SIZE), desc=f"PPL {row.Source}", unit="batch"):
            batch_records = valid_records[i:i + BATCH_SIZE]
            batch_seqs = [str(r.seq) for r in batch_records]

            try:
                batch_ppls = calculate_batch_ppl(batch_seqs, tokenizer, model, DEVICE, MAX_LENGTH)
                for j, record in enumerate(batch_records):
                    seq_str = str(record.seq).upper()
                    analysis = ProteinAnalysis(seq_str)
                    all_results.append({
                        "ID": record.id,
                        "Source": row.Source,
                        "Type": identify_histone_type(record.description),
                        "Length": len(seq_str),
                        "MW_kDa": analysis.molecular_weight() / 1000.0,
                        "pI": analysis.isoelectric_point(),
                        "PPL": batch_ppls[j],
                    })
            except RuntimeError as exc:
                if "out of memory" in str(exc):
                    print(f"[OOM] Batch {i} failed. Try reducing BATCH_SIZE={BATCH_SIZE}.")
                    torch.cuda.empty_cache()
                else:
                    print(f"[RuntimeError] Batch {i}: {exc}")
                continue
            except Exception as exc:
                print(f"[Error] Batch {i}: {exc}")
                continue

    return pd.DataFrame(all_results)


df_ppl = build_ppl_dataframe(manifest)
display(df_ppl.head())

In [ ]:
def plot_distributions_cns(df: pd.DataFrame, histone_type: str = "H1") -> None:
    subset = df[df["Histone_Type"] == histone_type].copy()
    if subset.empty:
        print(f"No CDS data available for {histone_type}.")
        return

    order = [k for k in PLOT_SOURCE_ORDER if k in subset["Dataset"].unique()]
    fig, axes = plt.subplots(1, 3, figsize=(22, 7), dpi=300)
    metrics = [
        ("Length", "CDS Length (bp)", axes[0]),
        ("GC_Content", "CDS GC Content (%)", axes[1]),
        ("GC3", "CDS GC3 Content (%)", axes[2]),
    ]

    for col, label, ax in metrics:
        sns.violinplot(data=subset, x="Dataset", y=col, order=order, palette=DATASET_PALETTE, ax=ax, saturation=0.8, linewidth=0, cut=0, inner=None)
        sns.boxplot(
            data=subset,
            x="Dataset",
            y=col,
            order=order,
            width=0.15,
            boxprops={"zorder": 2, "facecolor": "white", "edgecolor": "black", "alpha": 0.9, "linewidth": 1.2},
            whiskerprops={"color": "black", "linewidth": 1.2},
            capprops={"color": "black", "linewidth": 1.2},
            medianprops={"color": "black", "linewidth": 1.5},
            ax=ax,
            showfliers=False,
        )
        if col == "Length":
            ax.set_ylim(0, 2000)
        ax.set_xlabel("")
        ax.set_ylabel(label, fontsize=19, labelpad=12)
        ax.set_xticks(range(len(order)))
        formatted_labels = [
            lbl.replace("Temperature ", "Temp ").replace(" Top-p ", "\nTop-p ").replace("Natural Sequences", "Natural\nSequences")
            for lbl in order
        ]
        ax.set_xticklabels(formatted_labels, rotation=45, ha="right", rotation_mode="anchor", fontsize=15)
        ax.tick_params(axis="y", labelsize=16)
        sns.despine(ax=ax)
        ax.grid(False)

    plt.subplots_adjust(wspace=0.35, bottom=0.25)
    plt.show()


def plot_codon_usage_grid(df: pd.DataFrame, histone_type: str = "H1") -> None:
    subset = df[df["Histone_Type"] == histone_type].copy()
    if subset.empty:
        print(f"No CDS data available for {histone_type}.")
        return

    all_datasets = [ds for ds in PLOT_SOURCE_ORDER if ds in subset["Dataset"].unique()]
    dataset_freqs = {}
    for ds in all_datasets:
        sub_ds = subset[subset["Dataset"] == ds]
        counter = Counter()
        for seq in sub_ds["Sequence"]:
            codons = [seq[i:i + 3] for i in range(0, len(seq), 3) if len(seq[i:i + 3]) == 3]
            counter.update(codons)
        total = sum(counter.values()) or 1
        dataset_freqs[ds] = [counter.get(c, 0) / total for c in ORDERED_CODONS]

    baseline_key = "Natural Sequences" if "Natural Sequences" in dataset_freqs else all_datasets[0]
    nat_freqs = dataset_freqs[baseline_key]
    gen_datasets = [ds for ds in all_datasets if ds != baseline_key]

    nrows, ncols = 2, 2
    n_ds = len(gen_datasets)
    N = len(ORDERED_CODONS)
    theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
    theta_loop = np.append(theta, theta[0])
    nat_freqs_loop = np.append(nat_freqs, nat_freqs[0])
    width = (2 * np.pi) / N * 0.9

    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 10.5), subplot_kw={"projection": "polar"})
    axes_flat = axes.flatten()

    for i in range(nrows * ncols):
        ax = axes_flat[i]
        if i >= n_ds:
            ax.axis("off")
            continue

        ds = gen_datasets[i]
        gen_freqs = dataset_freqs[ds]
        color = DATASET_PALETTE.get(ds, "#fc8d62")
        ax.bar(theta, gen_freqs, width=width, bottom=0.0, color=color, alpha=0.5, zorder=2, label="Generated")
        ax.plot(theta_loop, nat_freqs_loop, color="#2c3e50", linewidth=1.5, linestyle="-", zorder=5, label="Natural Sequence")
        ax.grid(False)
        max_val = max(max(gen_freqs), max(nat_freqs)) * 1.1

        for r in np.linspace(0, max_val, 5)[1:]:
            ax.plot(np.linspace(0, 2 * np.pi, 100), [r] * 100, color="gray", alpha=0.1, linewidth=0.5, zorder=1)
        for b_idx in BOUNDARY_INDICES:
            angle = (b_idx * 2 * np.pi / N) - (np.pi / N)
            ax.plot([angle, angle], [0, max_val], color="gray", linestyle="--", linewidth=0.6, alpha=0.3, zorder=1)

        ax.set_title(ds, y=1.1, fontsize=15, color="black")
        ax.set_xticks(theta)
        ax.set_xticklabels(ORDERED_CODONS, fontsize=7, rotation=90)
        ax.set_ylim(0, max_val)
        ax.set_yticklabels([])
        if i == 0:
            ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.18), fontsize=11)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3, wspace=0.3)
    plt.show()


def plot_aa_composition(df_cds: pd.DataFrame, histone_type: str = "H1") -> None:
    subset = df_cds[df_cds["Histone_Type"] == histone_type].copy()
    if subset.empty:
        print(f"No CDS data available for {histone_type}.")
        return

    valid_aa = sorted("ACDEFGHIKLMNPQRSTVWY")
    plot_data = []
    datasets = [k for k in PLOT_SOURCE_ORDER if k in subset["Dataset"].unique()]

    for ds in datasets:
        sub_ds = subset[subset["Dataset"] == ds]
        counter = Counter()
        for seq in sub_ds["Sequence"]:
            try:
                valid_len = (len(seq) // 3) * 3
                prot = Seq(seq[:valid_len]).translate(to_stop=True)
                counter.update(str(prot))
            except Exception:
                pass
        total = sum(counter.values()) or 1
        for aa in valid_aa:
            plot_data.append({"Amino_Acid": aa, "Frequency": counter.get(aa, 0) / total, "Dataset": ds})

    df_plot = pd.DataFrame(plot_data)
    plt.figure(figsize=(15, 6.5), dpi=300)
    ax = sns.barplot(data=df_plot, x="Amino_Acid", y="Frequency", hue="Dataset", palette=DATASET_PALETTE, edgecolor="black", linewidth=0.8, alpha=0.85)
    ax.set_ylabel("Frequency", fontsize=21, labelpad=10)
    ax.set_xlabel("")
    ax.tick_params(axis="y", labelsize=16)
    ax.tick_params(axis="x", labelsize=16)
    sns.despine(ax=ax)
    ax.grid(False)

    if ax.get_legend() is not None:
        ax.legend_.remove()
    legend_elements = [
        mpatches.Patch(facecolor=DATASET_PALETTE[ds], edgecolor="black", linewidth=0.8, alpha=0.85, label=ds)
        for ds in datasets
    ]
    ax.legend(handles=legend_elements, loc="upper right", frameon=False, ncol=1, fontsize=17)
    plt.tight_layout()
    plt.show()


def plot_mw_pi(df_protein: pd.DataFrame) -> None:
    if df_protein.empty:
        print("No protein data available.")
        return

    plot_df = df_protein[df_protein["Predicted_Type"] != "Unknown"].copy()
    nat_mask = plot_df["Source"] == "Natural Sequences"
    nat_df = plot_df[nat_mask]
    other_df = plot_df[~nat_mask]
    if not nat_df.empty:
        nat_df = nat_df.groupby("Predicted_Type", group_keys=False).apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42)).reset_index(drop=True)
    plot_df = pd.concat([nat_df, other_df], ignore_index=True)

    sources = [k for k in PLOT_SOURCE_ORDER if k in plot_df["Source"].unique()]
    fig, axes = plt.subplots(1, len(sources), figsize=(22, 5), dpi=300)
    if len(sources) == 1:
        axes = [axes]

    hue_order = HISTONE_TYPES
    palette = sns.color_palette("Set1", n_colors=len(hue_order))
    for i, source in enumerate(sources):
        ax = axes[i]
        subset = plot_df[plot_df["Source"] == source]
        show_legend = i == len(sources) - 1
        sns.scatterplot(
            data=subset,
            x="MW_kDa",
            y="pI",
            hue="Predicted_Type",
            style="Predicted_Type",
            hue_order=hue_order,
            style_order=hue_order,
            s=40,
            alpha=0.75,
            edgecolor="white",
            linewidth=0.4,
            palette=palette,
            ax=ax,
            legend=show_legend,
        )
        ax.set_title(source, fontsize=16, fontweight="bold", pad=12)
        ax.axhline(y=10.0, color="#333333", linestyle="--", linewidth=1.2, alpha=0.5, zorder=1)
        ax.set_xlim(0, 40)
        ax.set_ylim(9, 12.5)
        if i == 0:
            ax.set_ylabel("Isoelectric Point (pI)", fontsize=17, labelpad=10)
        else:
            ax.set_ylabel("")
        ax.set_xlabel("Molecular Weight (kDa)", fontsize=17, labelpad=10)
        ax.tick_params(axis="both", labelsize=15)
        ax.tick_params(axis="both", which="major", direction="out", length=5, width=1.2, bottom=True, left=True)
        sns.despine(ax=ax)
        ax.grid(False)

    if axes[-1].get_legend() is not None:
        axes[-1].legend(title="Histone Type", title_fontsize=15, fontsize=15, bbox_to_anchor=(1.0, 1), loc="upper left", frameon=False, markerscale=1.5)
    plt.subplots_adjust(wspace=0.25)
    plt.show()


def plot_ppl_distribution(df_ppl: pd.DataFrame) -> None:
    if df_ppl.empty:
        print("No PPL data available.")
        return
    plt.figure(figsize=(10, 6))
    limit_ppl = df_ppl["PPL"].quantile(0.98)
    plot_df = df_ppl[df_ppl["PPL"] < limit_ppl]
    sns.violinplot(x="Source", y="PPL", data=plot_df, palette="muted", inner="quartile")
    sns.stripplot(x="Source", y="PPL", data=plot_df, color="k", alpha=0.1, size=2, jitter=True)
    plt.title(f"Distribution of Perplexity (PPL) by Source (Model: {MODEL_NAME.split('/')[-1]})", fontsize=14)
    plt.ylabel("Perplexity (Lower is Better)")
    plt.xlabel("Data Source")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.show()


def plot_ppl_by_type(df_ppl: pd.DataFrame) -> None:
    if df_ppl.empty:
        print("No PPL data available.")
        return
    histone_df = df_ppl[df_ppl["Type"] != "Unknown"].copy()
    if histone_df.empty:
        print("No typed PPL data available.")
        return
    limit_ppl_h = histone_df["PPL"].quantile(0.98)
    histone_df = histone_df[histone_df["PPL"] < limit_ppl_h]
    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
    sns.boxplot(x="Type", y="PPL", hue="Source", data=histone_df, palette="Set2", showfliers=False, linewidth=1.2, ax=ax, order=HISTONE_TYPES)
    ax.tick_params(axis="both", which="major", direction="out", length=5, width=1.2, bottom=True, left=True)
    ax.set_ylabel("ProGen2 Perplexity", labelpad=10, fontsize=16)
    ax.set_xlabel("Histone Type", labelpad=10, fontsize=16)
    plt.legend(fontsize=12, frameon=False, bbox_to_anchor=(0.7, 1), loc="upper left")
    sns.despine(ax=ax)
    ax.grid(False)
    plt.show()


set_nature_style()
plot_distributions_cns(df_cds, TARGET)
plot_codon_usage_grid(df_cds, TARGET)
plot_aa_composition(df_cds, TARGET)
plot_mw_pi(df_protein)
plot_ppl_distribution(df_ppl)
plot_ppl_by_type(df_ppl)